In [ ]:
# =============================================================================
# NOTEBOOK: 04 — Stability Lobe Diagrams (SLD)
# Project : Chatter Detection in CNC Machining
# Author  : Data Science / Mechanical Engineering Team
# =============================================================================
# Run this file directly (`python 04_stability_lobe_diagrams.py`) or copy
# each section into a Jupyter notebook cell.
# =============================================================================


In [ ]:
# ── CELL 1 ── Imports & global style
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

pio.templates.default = "plotly_dark"   # dark theme throughout

DATA_PATH  = Path("../data/processed/FINAL_ML_DATASET.csv")
EXPORT_DIR = Path("../figures")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("✅  Imports loaded.")

✅  Imports loaded.


In [9]:
# ── CELL 2 ── Physical System Parameters
# ─────────────────────────────────────────────────────────────────────────────
# 1-DOF regenerative chatter model (Altintas-Budak, 1995)
# All units: SI (Pa, kg, m, rad/s).  DOC results displayed in µm.

fn      = 97.7          # Natural frequency           [Hz]
zeta    = 0.05          # Damping ratio               [dimensionless]
Kf      = 2_000e6       # Cutting force coefficient   [Pa]  (2000 MPa)
m_modal = 10.0          # Modal mass                  [kg]

omega_n = 2 * np.pi * fn             # [rad/s]
k_stiff = m_modal * omega_n**2       # Modal stiffness [N/m]
c_damp  = 2 * zeta * omega_n * m_modal  # Modal damping [N*s/m]

N_TEETH = 2     # Number of cutting teeth (set to 1 for turning)
N_LOBES = 12    # Number of stability lobes to plot

print(f"Natural frequency   wn = {fn} Hz  ({omega_n:.2f} rad/s)")
print(f"Damping ratio        z = {zeta}")
print(f"Cutting coefficient Kf = {Kf/1e6:.0f} MPa")
print(f"Modal mass           m = {m_modal} kg")
print(f"Modal stiffness      k = {k_stiff/1e6:.3f} MN/m")
print(f"Cutting teeth        N = {N_TEETH}")

Natural frequency   wn = 97.7 Hz  (613.87 rad/s)
Damping ratio        z = 0.05
Cutting coefficient Kf = 2000 MPa
Modal mass           m = 10.0 kg
Modal stiffness      k = 3.768 MN/m
Cutting teeth        N = 2


In [ ]:
# ── CELL 3 ── Transfer Function & Stability Lobe Computation
# ─────────────────────────────────────────────────────────────────────────────

def frf_1dof(omega_c, omega_n, zeta, k):
    """
    1-DOF Frequency Response Function (FRF).

    G(jw) = 1 / [k * ((1 - r^2) + 2j*z*r)]   where r = w/wn

    Returns real and imaginary parts of G.
    """
    r     = omega_c / omega_n
    denom = k * ((1 - r**2)**2 + (2 * zeta * r)**2)
    G_real = (1 - r**2) / denom
    G_imag = -(2 * zeta * r) / denom
    return G_real, G_imag


def compute_stability_lobes(
    fn, zeta, Kf, m_modal,
    N_teeth=2, n_lobes=12, n_pts=4000,
    rpm_min=50.0, rpm_max=30000.0,
):
    """
    Generate Stability Lobe Diagram data using the 1-DOF analytical method
    (Altintas & Budak, Int. J. Machine Tools Manuf., 1995).

    For each chatter frequency wc > wn (where Re[G] < 0):
        a_lim [m]  = -1 / (2 * Kf * Re[G(j*wc)])
        psi   [rad]= pi - 2*arctan2(Im[G], Re[G])
        RPM_k      = 60 * wc / (N_teeth * (2*pi*k + psi))   k = 0,1,2,...

    Parameters
    ----------
    fn      : Natural frequency [Hz]
    zeta    : Damping ratio
    Kf      : Cutting force coefficient [Pa]
    m_modal : Modal mass [kg]
    N_teeth : Number of cutting teeth
    n_lobes : Number of lobes to compute
    n_pts   : Resolution of wc sweep
    rpm_min : Lower RPM bound for output
    rpm_max : Upper RPM bound for output

    Returns
    -------
    pd.DataFrame with columns: RPM, a_lim_um, a_lim_mm, lobe_id
    """
    wn       = 2 * np.pi * fn
    k_spring = m_modal * wn**2

    omega_sweep = np.linspace(wn * 1.0005, wn * (n_lobes + 1.5), n_pts)
    G_real, G_imag = frf_1dof(omega_sweep, wn, zeta, k_spring)

    # Only frequencies where Re[G] < 0 give a finite positive a_lim
    mask    = G_real < 0
    omega_c = omega_sweep[mask]
    Gr      = G_real[mask]
    Gi      = G_imag[mask]

    records = []
    for k in range(n_lobes):
        a_lim = -1.0 / (2.0 * Kf * Gr)            # [m], positive

        psi = np.pi - 2.0 * np.arctan2(Gi, Gr)
        psi = np.where(psi <= 0, psi + 2 * np.pi, psi)   # ensure psi in (0, 2pi]

        period = (2 * np.pi * k + psi) / omega_c   # tooth period [s]
        rpm    = 60.0 / (N_teeth * period)          # [RPM]

        for r, a, in zip(rpm, a_lim):
            if rpm_min <= r <= rpm_max and a > 0:
                records.append({
                    "RPM":      r,
                    "a_lim_um": a * 1e6,
                    "a_lim_mm": a * 1e3,
                    "lobe_id":  k,
                })

    df = pd.DataFrame(records)
    df = df.sort_values(["lobe_id", "RPM"]).reset_index(drop=True)
    return df


df_lobes = compute_stability_lobes(fn, zeta, Kf, m_modal, N_TEETH, N_LOBES)
print(f"✅  Stability lobes computed: {len(df_lobes):,} points across {N_LOBES} lobes.")
print(df_lobes.groupby("lobe_id")["RPM"].agg(["min", "max"]).to_string())

✅  Stability lobes computed: 48,000 points across 12 lobes.
                 min           max
lobe_id                           
0        2394.566942  26420.759172
1        1365.002585  15842.423789
2         948.057001  11312.948834
3         723.680705   8797.635016
4         583.807419   7197.375210
5         488.296786   6089.682695
6         418.733287   5277.468254
7         366.412438   4656.416413
8         325.714335   4166.144965
9         293.153263   3769.279645
10        266.510674   3441.448612
11        244.307339   3166.080496


In [11]:
# ── CELL 4 ── Load & Aggregate Experimental Data
# ─────────────────────────────────────────────────────────────────────────────

def load_experimental_data(path):
    """
    Load FINAL_ML_DATASET.csv and aggregate repeated measurement windows.

    Convention: a condition (RPM, DOC_um) is labelled CHATTER (1) if ANY
    measurement window at that condition shows chatter (conservative / safe).
    """
    df = pd.read_csv(path)
    required = {"RPM", "DOC_um", "Label"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV missing columns: {missing}")

    df_agg = (
        df.groupby(["RPM", "DOC_um"])
          .agg(
              Label        = ("Label", "max"),
              n_windows    = ("Label", "count"),
              chatter_frac = ("Label", "mean"),
          )
          .reset_index()
    )
    df_agg["DOC_mm"] = df_agg["DOC_um"] / 1000.0
    return df_agg


df_exp = load_experimental_data(DATA_PATH)

n_stable  = (df_exp["Label"] == 0).sum()
n_chatter = (df_exp["Label"] == 1).sum()
print(f"✅  Experimental data loaded: {len(df_exp)} unique operating conditions")
print(f"    Stable  : {n_stable}")
print(f"    Chatter : {n_chatter}")
print(f"    RPM  range : {df_exp['RPM'].min():.0f} - {df_exp['RPM'].max():.0f} RPM")
print(f"    DOC  range : {df_exp['DOC_um'].min():.0f} - {df_exp['DOC_um'].max():.0f} um")

✅  Experimental data loaded: 49 unique operating conditions
    Stable  : 1
    Chatter : 48
    RPM  range : 52 - 550 RPM
    DOC  range : 100 - 2600 um


In [ ]:
# ── CELL 5 ── Design Tokens & Trace Builders
# ─────────────────────────────────────────────────────────────────────────────

STABLE_COLOR  = "#00E5A0"   # Teal-green
CHATTER_COLOR = "#FF4C6E"   # Vivid red-pink
LOBE_COLOR    = "#A78BFA"   # Lavender-purple
BG_COLOR      = "#0F0F1A"   # Near-black navy


def make_exp_traces(df_exp, marker_size=10):
    """Plotly scatter traces for stable and chatter experimental conditions."""
    stable  = df_exp[df_exp["Label"] == 0]
    chatter = df_exp[df_exp["Label"] == 1]

    tmpl = (
        "<b>%{fullData.name}</b><br>"
        "Spindle Speed : <b>%{x:.0f} RPM</b><br>"
        "Depth of Cut  : <b>%{y:.0f} um</b><br>"
        "Windows       : %{customdata[0]:.0f}<br>"
        "Chatter %%    : %{customdata[1]:.1f}%%"
        "<extra></extra>"
    )

    tr_stable = go.Scatter(
        x=stable["RPM"], y=stable["DOC_um"], mode="markers", name="Stable",
        customdata=np.stack([stable["n_windows"], stable["chatter_frac"]*100], axis=-1),
        hovertemplate=tmpl,
        marker=dict(symbol="circle", size=marker_size, color=STABLE_COLOR,
                    opacity=0.85, line=dict(color="white", width=0.5)),
    )
    tr_chatter = go.Scatter(
        x=chatter["RPM"], y=chatter["DOC_um"], mode="markers", name="Chatter",
        customdata=np.stack([chatter["n_windows"], chatter["chatter_frac"]*100], axis=-1),
        hovertemplate=tmpl,
        marker=dict(symbol="x", size=marker_size+2, color=CHATTER_COLOR,
                    opacity=0.90, line=dict(color=CHATTER_COLOR, width=1.5)),
    )
    return tr_stable, tr_chatter


def make_lobe_traces(df_lobes, rpm_min=None, rpm_max=None):
    """One Plotly line trace per stability lobe."""
    traces = []
    df = df_lobes.copy()
    if rpm_min is not None:
        df = df[df["RPM"].between(rpm_min, rpm_max)]
    for k, grp in df.groupby("lobe_id"):
        grp = grp.sort_values("RPM")
        traces.append(go.Scatter(
            x=grp["RPM"], y=grp["a_lim_um"], mode="lines",
            name="Stability Boundary" if k == 0 else f"Lobe {k}",
            showlegend=(k == 0), legendgroup="lobes",
            line=dict(color=LOBE_COLOR, width=2.5 if k == 0 else 1.5,
                      dash="solid" if k == 0 else "dot"),
            hovertemplate=(
                f"<b>Lobe {k} - Stability Boundary</b><br>"
                "Spindle Speed : <b>%{x:.0f} RPM</b><br>"
                "a_lim         : <b>%{y:.0f} um  (%{y:.3f} mm)</b>"
                "<extra></extra>"
            ),
        ))
    return traces


def build_envelope(df_lobes, rpm_max):
    """Lower stability envelope across all lobes."""
    return (
        df_lobes[df_lobes["RPM"] <= rpm_max]
        .groupby("RPM")["a_lim_um"].min()
        .reset_index().sort_values("RPM")
    )

In [7]:
# ── CELL 6 ── Figure 1: Full-Range Stability Lobe Diagram
# ─────────────────────────────────────────────────────────────────────────────

RPM_FULL_MAX = max(df_exp["RPM"].max() * 3, 5000)

fig_full = go.Figure()

# Theoretical lobes
for tr in make_lobe_traces(df_lobes, rpm_max=RPM_FULL_MAX):
    fig_full.add_trace(tr)

# Shaded stable region under envelope
df_env = build_envelope(df_lobes, RPM_FULL_MAX)
fig_full.add_trace(go.Scatter(
    x=pd.concat([df_env["RPM"], df_env["RPM"][::-1]]),
    y=pd.concat([df_env["a_lim_um"], pd.Series([0] * len(df_env))]),
    fill="toself", fillcolor="rgba(0,229,160,0.07)",
    line=dict(color="rgba(0,0,0,0)"),
    name="Stable Region", hoverinfo="skip",
))

# Experimental scatter
tr_s, tr_c = make_exp_traces(df_exp)
fig_full.add_trace(tr_s)
fig_full.add_trace(tr_c)

# Natural frequency reference
fig_full.add_vline(
    x=fn * 60, line=dict(color="rgba(255,220,50,0.45)", width=1.5, dash="dash"),
    annotation_text=f"  fn = {fn} Hz",
    annotation_font=dict(color="#FFDC32", size=11),
    annotation_position="top right",
)

fig_full.update_layout(
    title=dict(
        text=(
            "<b>Stability Lobe Diagram - CNC Machining Chatter Detection</b><br>"
            f"<sup>1-DOF Altintas-Budak Model  |  fn={fn} Hz, z={zeta}, "
            f"Kf={Kf/1e6:.0f} MPa, m={m_modal} kg,  N_teeth={N_TEETH}</sup>"
        ),
        font=dict(size=18), x=0.5,
    ),
    xaxis=dict(title="<b>Spindle Speed [RPM]</b>", title_font_size=14,
               gridcolor="rgba(255,255,255,0.07)", zeroline=False,
               range=[0, RPM_FULL_MAX]),
    yaxis=dict(title="<b>Axial Depth of Cut  a_lim  [um]</b>", title_font_size=14,
               gridcolor="rgba(255,255,255,0.07)", zeroline=False, rangemode="tozero"),
    legend=dict(bgcolor="rgba(15,15,26,0.8)", bordercolor="rgba(255,255,255,0.15)",
                borderwidth=1, font=dict(size=12)),
    paper_bgcolor=BG_COLOR, plot_bgcolor=BG_COLOR,
    font=dict(family="Inter, Arial, sans-serif", color="white"),
    hovermode="closest", width=1200, height=680,
    margin=dict(l=70, r=30, t=100, b=70),
)

fig_full.write_html(str(EXPORT_DIR / "SLD_full_range.html"))
print("✅  Full-range SLD saved -> figures/SLD_full_range.html")
fig_full.show()

✅  Full-range SLD saved -> figures/SLD_full_range.html


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# ── CELL 7 ── Figure 2: Zoomed View on Experimental RPM Range
# ─────────────────────────────────────────────────────────────────────────────

RPM_ZOOM_MIN = max(df_exp["RPM"].min() * 0.85, 50)
RPM_ZOOM_MAX = df_exp["RPM"].max() * 1.15
DOC_ZOOM_MAX = df_exp["DOC_um"].max() * 1.25

fig_zoom = make_subplots(
    rows=1, cols=2,
    column_widths=[0.68, 0.32],
    subplot_titles=[
        "<b>Zoomed: Experimental RPM Range</b>",
        "<b>Chatter Fraction by Condition</b>",
    ],
    horizontal_spacing=0.08,
)

# ── Left panel: zoomed SLD ────────────────────────────────────────────────────
for tr in make_lobe_traces(df_lobes, rpm_min=RPM_ZOOM_MIN, rpm_max=RPM_ZOOM_MAX):
    fig_zoom.add_trace(tr, row=1, col=1)

df_env_z = build_envelope(df_lobes, RPM_ZOOM_MAX)
df_env_z = df_env_z[df_env_z["RPM"] >= RPM_ZOOM_MIN]
if not df_env_z.empty:
    fig_zoom.add_trace(go.Scatter(
        x=pd.concat([df_env_z["RPM"], df_env_z["RPM"][::-1]]),
        y=pd.concat([df_env_z["a_lim_um"], pd.Series([0] * len(df_env_z))]),
        fill="toself", fillcolor="rgba(0,229,160,0.08)",
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False, hoverinfo="skip",
    ), row=1, col=1)

df_s_z = df_exp[(df_exp["Label"] == 0) & df_exp["RPM"].between(RPM_ZOOM_MIN, RPM_ZOOM_MAX)]
df_c_z = df_exp[(df_exp["Label"] == 1) & df_exp["RPM"].between(RPM_ZOOM_MIN, RPM_ZOOM_MAX)]

fig_zoom.add_trace(go.Scatter(
    x=df_s_z["RPM"], y=df_s_z["DOC_um"], mode="markers", name="Stable",
    marker=dict(symbol="circle", size=12, color=STABLE_COLOR, opacity=0.85,
                line=dict(color="white", width=0.5)),
    hovertemplate="<b>Stable</b><br>RPM=%{x:.0f}  DoC=%{y:.0f} um<extra></extra>",
), row=1, col=1)

fig_zoom.add_trace(go.Scatter(
    x=df_c_z["RPM"], y=df_c_z["DOC_um"], mode="markers", name="Chatter",
    marker=dict(symbol="x", size=14, color=CHATTER_COLOR, opacity=0.90,
                line=dict(color=CHATTER_COLOR, width=2)),
    hovertemplate="<b>Chatter</b><br>RPM=%{x:.0f}  DoC=%{y:.0f} um<extra></extra>",
), row=1, col=1)

# ── Right panel: chatter-fraction bubble chart ────────────────────────────────
df_bubble = df_exp[df_exp["RPM"].between(RPM_ZOOM_MIN, RPM_ZOOM_MAX)].copy()
df_bubble["chatter_pct"] = df_bubble["chatter_frac"] * 100

fig_zoom.add_trace(go.Scatter(
    x=df_bubble["RPM"], y=df_bubble["DOC_um"], mode="markers",
    showlegend=False,
    marker=dict(
        size=df_bubble["chatter_pct"] / 2 + 6,
        color=df_bubble["chatter_pct"],
        colorscale=[[0.0, STABLE_COLOR], [0.5, "#FFC55A"], [1.0, CHATTER_COLOR]],
        colorbar=dict(title="Chatter %", thickness=12, len=0.65, x=1.02),
        opacity=0.80,
        line=dict(color="white", width=0.4),
    ),
    hovertemplate="RPM=%{x:.0f}<br>DoC=%{y:.0f} um<br>Chatter=%{marker.color:.1f}%<extra></extra>",
), row=1, col=2)

# Axes
for col, xrng in [(1, [RPM_ZOOM_MIN, RPM_ZOOM_MAX]),
                  (2, [RPM_ZOOM_MIN, RPM_ZOOM_MAX])]:
    fig_zoom.update_xaxes(title_text="<b>Spindle Speed [RPM]</b>",
                          gridcolor="rgba(255,255,255,0.07)", zeroline=False,
                          range=xrng, row=1, col=col)
    fig_zoom.update_yaxes(title_text="<b>DoC [um]</b>",
                          gridcolor="rgba(255,255,255,0.07)", zeroline=False,
                          rangemode="tozero", range=[0, DOC_ZOOM_MAX], row=1, col=col)

fig_zoom.update_layout(
    title=dict(
        text=(
            "<b>Chatter Detection - Zoomed Stability Lobe Diagram</b><br>"
            f"<sup>Experimental data overlaid on 1-DOF boundary "
            f"(fn={fn} Hz, z={zeta}, Kf={Kf/1e6:.0f} MPa)</sup>"
        ),
        font=dict(size=16), x=0.5,
    ),
    paper_bgcolor=BG_COLOR, plot_bgcolor=BG_COLOR,
    font=dict(family="Inter, Arial, sans-serif", color="white"),
    legend=dict(bgcolor="rgba(15,15,26,0.8)", bordercolor="rgba(255,255,255,0.2)",
                borderwidth=1, font=dict(size=12)),
    hovermode="closest", width=1400, height=620,
    margin=dict(l=70, r=100, t=100, b=70),
)

fig_zoom.write_html(str(EXPORT_DIR / "SLD_zoomed.html"))
print("✅  Zoomed SLD saved -> figures/SLD_zoomed.html")
fig_zoom.show()

In [ ]:
# ── CELL 8 ── Figure 3: Interactive Dashboard with Range Slider
# ─────────────────────────────────────────────────────────────────────────────

def build_sld_dashboard(df_lobes, df_exp, fn, zeta, Kf, m_modal, N_teeth):
    """
    Full publication-quality interactive SLD dashboard.
    Includes: theoretical lobes, stable-zone fill, experimental scatter,
    range slider, spike lines, and key annotations.
    """
    fig    = go.Figure()
    RPM_MAX = max(df_exp["RPM"].max() * 3, 5000)

    # Stable zone fill
    df_e = build_envelope(df_lobes, RPM_MAX)
    fig.add_trace(go.Scatter(
        x=pd.concat([df_e["RPM"], df_e["RPM"][::-1]]),
        y=pd.concat([df_e["a_lim_um"], pd.Series([0] * len(df_e))]),
        fill="toself", fillcolor="rgba(0,229,160,0.07)",
        line=dict(color="rgba(0,0,0,0)"),
        name="Stable Zone", hoverinfo="skip",
    ))

    # Stability lobes
    for k, grp in df_lobes[df_lobes["RPM"] <= RPM_MAX].groupby("lobe_id"):
        grp = grp.sort_values("RPM")
        fig.add_trace(go.Scatter(
            x=grp["RPM"], y=grp["a_lim_um"], mode="lines",
            name="Stability Boundary" if k == 0 else f"Lobe {k}",
            showlegend=(k == 0), legendgroup="lobes",
            line=dict(color=LOBE_COLOR, width=3 if k == 0 else 1.8,
                      dash="solid" if k == 0 else "dot"),
            hovertemplate=(
                f"<b>Lobe {k}</b><br>"
                "Spindle Speed : %{x:.0f} RPM<br>"
                "a_lim         : %{y:.0f} um (%{y:.3f} mm)"
                "<extra></extra>"
            ),
        ))

    # Experimental data
    for df_c, name, color, symbol in [
        (df_exp[df_exp["Label"] == 0], "Stable",  STABLE_COLOR,  "circle"),
        (df_exp[df_exp["Label"] == 1], "Chatter", CHATTER_COLOR, "x"),
    ]:
        fig.add_trace(go.Scatter(
            x=df_c["RPM"], y=df_c["DOC_um"], mode="markers", name=name,
            marker=dict(symbol=symbol, size=11, color=color, opacity=0.88,
                        line=dict(color="white", width=0.5)),
            hovertemplate=(
                f"<b>{name}</b><br>"
                "RPM : %{x:.0f}<br>"
                "DoC : %{y:.0f} um  (%{y:.3f} mm)"
                "<extra></extra>"
            ),
        ))

    # Annotations
    a_lim_min = df_e["a_lim_um"].min()
    fig.add_hline(
        y=a_lim_min,
        line=dict(color="rgba(255,220,50,0.35)", width=1.2, dash="dash"),
        annotation_text=f"  Unconditional limit ~ {a_lim_min:.0f} um",
        annotation_font=dict(color="#FFDC32", size=10),
        annotation_position="bottom right",
    )
    fig.add_vline(
        x=fn * 60,
        line=dict(color="rgba(255,220,50,0.3)", width=1.2, dash="dash"),
        annotation_text=f"  fn = {fn} Hz",
        annotation_font=dict(color="#FFDC32", size=10),
        annotation_position="top right",
    )

    fig.update_layout(
        title=dict(
            text=(
                "<b>Stability Lobe Diagram - CNC Machining</b><br>"
                f"<sup>1-DOF Model  |  fn={fn} Hz  |  z={zeta}  |  "
                f"Kf={Kf/1e6:.0f} MPa  |  m={m_modal} kg  |  N={N_teeth} teeth</sup>"
            ),
            font=dict(size=20), x=0.5,
        ),
        xaxis=dict(
            title="<b>Spindle Speed [RPM]</b>", title_font_size=15,
            gridcolor="rgba(255,255,255,0.07)", zeroline=False,
            showspikes=True, spikecolor="rgba(200,200,200,0.4)",
            rangeslider=dict(visible=True, thickness=0.06, bgcolor="#1A1A2E"),
        ),
        yaxis=dict(
            title="<b>Critical Axial Depth of Cut  a_lim  [um]</b>",
            title_font_size=15,
            gridcolor="rgba(255,255,255,0.07)", zeroline=False,
            showspikes=True, spikecolor="rgba(200,200,200,0.4)",
            rangemode="tozero",
        ),
        legend=dict(bgcolor="rgba(15,15,26,0.9)", bordercolor="rgba(255,255,255,0.2)",
                    borderwidth=1, font=dict(size=13), itemsizing="constant"),
        paper_bgcolor=BG_COLOR, plot_bgcolor=BG_COLOR,
        font=dict(family="Inter, Arial, sans-serif", color="white"),
        hovermode="x unified",
        width=1300, height=720,
        margin=dict(l=80, r=40, t=110, b=90),
    )
    return fig


fig_dash = build_sld_dashboard(df_lobes, df_exp, fn, zeta, Kf, m_modal, N_TEETH)
fig_dash.write_html(str(EXPORT_DIR / "SLD_dashboard.html"))
print("✅  Interactive dashboard saved -> figures/SLD_dashboard.html")
fig_dash.show()

In [ ]:
# ── CELL 9 ── Summary Statistics & Theory vs. Experiment Comparison
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("  STABILITY BOUNDARY - KEY METRICS")
print("="*65)

df_env_full = build_envelope(df_lobes, rpm_max=df_lobes["RPM"].max())
a_lim_unconditional = df_env_full["a_lim_um"].min()
print(f"\n  Unconditional stability limit  : {a_lim_unconditional:.1f} um"
      f"  ({a_lim_unconditional/1000:.3f} mm)")

rpm_lo = df_exp["RPM"].min()
rpm_hi = df_exp["RPM"].max()
df_local = df_env_full[df_env_full["RPM"].between(rpm_lo, rpm_hi)]
if not df_local.empty:
    print(f"\n  Experimental RPM range         : {rpm_lo:.0f} - {rpm_hi:.0f} RPM")
    print(f"  a_lim min in range             : {df_local['a_lim_um'].min():.1f} um"
          f"  ({df_local['a_lim_um'].min()/1000:.3f} mm)")
    print(f"  a_lim max in range             : {df_local['a_lim_um'].max():.1f} um"
          f"  ({df_local['a_lim_um'].max()/1000:.3f} mm)")

# Fraction of chatter points correctly above the theoretical boundary
chatter_pts = df_exp[df_exp["Label"] == 1]
if not chatter_pts.empty:
    correct = 0
    for _, row in chatter_pts.iterrows():
        idx       = (df_env_full["RPM"] - row["RPM"]).abs().idxmin()
        threshold = df_env_full.loc[idx, "a_lim_um"]
        if row["DOC_um"] > threshold:
            correct += 1
    print(f"\n  Chatter pts above theory boundary: "
          f"{correct}/{len(chatter_pts)}  ({correct/len(chatter_pts)*100:.1f}%)")

# Stable points correctly below boundary
stable_pts = df_exp[df_exp["Label"] == 0]
if not stable_pts.empty:
    correct_s = 0
    for _, row in stable_pts.iterrows():
        idx       = (df_env_full["RPM"] - row["RPM"]).abs().idxmin()
        threshold = df_env_full.loc[idx, "a_lim_um"]
        if row["DOC_um"] <= threshold:
            correct_s += 1
    print(f"  Stable  pts below  theory boundary: "
          f"{correct_s}/{len(stable_pts)}  ({correct_s/len(stable_pts)*100:.1f}%)")

print("\n" + "="*65)
print("  OUTPUT FILES")
print("="*65)
for f in sorted(EXPORT_DIR.glob("*.html")):
    print(f"   {f.name}")

print("\n✅  Notebook 04 complete.")